# DATA SET AUGMENTATION AND SPLIT

- For Crackforest dataset due to its small size
- Use for segmentation experiments

### Import and Config path

In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import albumentations as A


IMAGE_DIR = "data/raw/crackforest/Images"
MASK_DIR  = "data/raw/crackforest/Masks"

OUTPUT_DIR = "data/processed/crackforest_augmented_50_15_35"

AUGMENT_TIMES = 15

TRAIN_RATIO = 0.50
VAL_RATIO   = 0.15
TEST_RATIO  = 0.35

c:\Users\hiend\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# =========================
# 4. Create folder structure
# =========================
splits = ["train", "val", "test"]

for split in splits:
    os.makedirs(f"{OUTPUT_DIR}/{split}/Images", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/{split}/Masks", exist_ok=True)

# =========================
# 5. Load file names
# =========================
image_files = sorted(os.listdir(IMAGE_DIR))
mask_files  = sorted(os.listdir(MASK_DIR))

assert len(image_files) == len(mask_files), "Mismatch in dataset!"

# =========================
# 6. Train/Val/Test split
# =========================
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    image_files, mask_files,
    test_size=(1 - TRAIN_RATIO),
    random_state=42
)

# Step 2
val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs, temp_masks,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state=42
)

print(f"Train: {len(train_imgs)}")
print(f"Val:   {len(val_imgs)}")
print(f"Test:  {len(test_imgs)}")

# =========================
# 7. Augmentation pipeline
# =========================
train_transform = A.Compose([
    A.OneOf([
        A.HorizontalFlip(p=1),
        A.VerticalFlip(p=1),
        A.RandomRotate90(p=1),
    ], p=0.8),

    A.ShiftScaleRotate(
        shift_limit=0.1,
        scale_limit=0.3,
        rotate_limit=30,
        p=0.8
    ),

    A.OneOf([
        A.RandomBrightnessContrast(),
        A.CLAHE(),
        A.RandomGamma(),
    ], p=0.7),

    A.OneOf([
        A.GaussNoise(),
        A.MotionBlur(),
        A.MedianBlur(),
    ], p=0.5),

    A.ElasticTransform(alpha=1, sigma=50, p=0.3),
])

# No augmentation for val/test
val_test_transform = A.Compose([])

# =========================
# 8. Processing function
# =========================
def process_split(images, masks, split, transform, augment_times=1):
    for idx, (img_name, mask_name) in enumerate(
        tqdm(zip(images, masks), total=len(images))
    ):
        img_path = os.path.join(IMAGE_DIR, img_name)
        mask_path = os.path.join(MASK_DIR, mask_name)

        image = cv2.imread(img_path)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Create base ID like 001, 002, ...
        base_id = f"{idx:03d}"

        # ---- Save original ----
        img_filename = f"{base_id}.jpg"
        mask_filename = f"{base_id}_label.PNG"

        cv2.imwrite(f"{OUTPUT_DIR}/{split}/Images/{img_filename}", image)
        cv2.imwrite(f"{OUTPUT_DIR}/{split}/Masks/{mask_filename}", mask)

        # ---- Augmentations ----
        for i in range(augment_times):
            augmented = transform(image=image, mask=mask)
            aug_img = augmented["image"]
            aug_mask = augmented["mask"]

            aug_img_filename = f"{base_id}_aug{i}.jpg"
            aug_mask_filename = f"{base_id}_aug{i}_label.PNG"

            cv2.imwrite(f"{OUTPUT_DIR}/{split}/Images/{aug_img_filename}", aug_img)
            cv2.imwrite(f"{OUTPUT_DIR}/{split}/Masks/{aug_mask_filename}", aug_mask)

# =========================
# 9. Run processing
# =========================
print("Processing TRAIN...")
process_split(train_imgs, train_masks, "train", train_transform, augment_times=AUGMENT_TIMES)

print("Processing VAL...")
process_split(val_imgs, val_masks, "val", val_test_transform, augment_times=0)

print("Processing TEST...")
process_split(test_imgs, test_masks, "test", val_test_transform, augment_times=0)

print("Done!")

c:\Users\hiend\AppData\Local\Programs\Python\Python313\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Train: 59
Val:   17
Test:  42
Processing TRAIN...


100%|██████████| 59/59 [00:16<00:00,  3.51it/s]


Processing VAL...


100%|██████████| 17/17 [00:00<00:00, 137.17it/s]


Processing TEST...


100%|██████████| 42/42 [00:00<00:00, 134.09it/s]

Done!
